# CSE 151B — Optimized SFT + GRPO + Inference (Colab Pro A100, fresh start)

This notebook is self-contained for a fresh Colab runtime. It assumes **nothing is already in Google Drive**.

You will upload:
- `public.jsonl`
- `judger.py`
- `utils.py`

Then the notebook trains task-specific LoRA adapters:
- free-form math adapter
- MCQ adapter

Run cells top to bottom. Restart runtime after Cell 1.


## Cell 1 — Install dependencies
**Restart runtime after this cell.**

In [ ]:
# Install dependencies for Colab A100.
# After this cell finishes, the runtime will restart automatically.
!pip install -q -U \
    'transformers>=4.51.0' \
    'trl>=0.17.0' \
    'peft>=0.14.0' \
    'accelerate>=1.4.0' \
    bitsandbytes \
    datasets \
    sympy \
    antlr4-python3-runtime==4.11.1 \
    tqdm

import os
os.kill(os.getpid(), 9)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 35.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.0 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.1 which is incompatible.


## Cell 2 — Mount Drive & configure paths

In [ ]:
# Fresh Colab setup: no Drive files assumed.
# You can keep everything in the Colab session under /content.
# Optional: set SAVE_TO_DRIVE=True later if you want persistence.

import os
from pathlib import Path

WORK_DIR    = '/content/cse151b'
DATA_PATH   = f'{WORK_DIR}/public.jsonl'
OUTPUT_DIR  = '/content/cse151b_outputs/Qwen/qwen3-task-adapters'
RESULTS_DIR = '/content/cse151b_outputs/results'

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

%cd {WORK_DIR}
print('Working directory:', os.getcwd())
print('Dataset path:', DATA_PATH)
print('Output directory:', OUTPUT_DIR)
print('Results directory:', RESULTS_DIR)


/content/cse151b
Working directory: /content/cse151b
Dataset path: /content/cse151b/public.jsonl
Output directory: /content/cse151b_outputs/Qwen/qwen3-task-adapters
Results directory: /content/cse151b_outputs/results


## Cell 3 — Copy helper files to working directory

In [ ]:
# Upload required files into the current Colab session.
# Run this cell and choose:
#   public.jsonl
#   judger.py
#   utils.py
#
# If you already uploaded them to /content/cse151b, this cell will skip upload.

from pathlib import Path
import shutil, os

required = ['public.jsonl', 'judger.py', 'utils.py']
missing = [f for f in required if not Path(WORK_DIR, f).exists()]

if missing:
    print('Missing files:', missing)
    print('Upload the missing files now.')
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        src = Path(name)
        dst = Path(WORK_DIR) / src.name
        if src.resolve() != dst.resolve():
            shutil.move(str(src), str(dst))
        print(f'Uploaded -> {dst}')
else:
    print('All required files already exist.')

print('\nCurrent working directory files:')
!ls -lah {WORK_DIR}


All required files already exist.

Current working directory files:
total 716K
drwxr-xr-x 2 root root 4.0K May 30 23:20 .
drwxr-xr-x 1 root root 4.0K May 30 23:20 ..
-rw-r--r-- 1 root root  38K May 30 23:20 judger.py
-rw-r--r-- 1 root root 653K May 30 23:19 public.jsonl
-rw-r--r-- 1 root root  12K May 30 23:20 utils.py


## Cell 4 — Imports & global config

In [ ]:
import os, re, json, gc, sys
from pathlib import Path
from typing import List, Optional, Tuple, Any
from tqdm.auto import tqdm

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainingArguments,
    Trainer, DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

try:
    from trl import GRPOConfig, GRPOTrainer
except Exception as e:
    print(f'TRL import warning: {e}')
    GRPOConfig = GRPOTrainer = None

#sys.path.insert(0, WORK_DIR)
#try:
from judger import Judger
"""except Exception as e:
    print(f'Judger import warning: {e}')
    Judger = None"""

# ── Self-contained prompt construction ────────────────────────────────────
SYSTEM_PROMPT_MATH = """You are an expert math tutor. Solve the problem accurately and concisely.

Use this structure:
Step 1: Understand — state the given information and what is being asked.
Step 2: Plan — identify the formula, theorem, or strategy.
Step 3: Solve — show the necessary algebra/calculation steps.
Step 4: Verify — briefly check the answer.

Put exactly one final answer at the end in \\boxed{}.

Rules:
- Prefer exact answers when appropriate.
- Use decimals only when required; then give sufficient precision.
- For multiple sub-answers, put them in one box separated by commas.
- Use plain-text math notation inside the box: *, ^, sqrt(), pi, parentheses.
- Avoid unnecessary commentary.
"""

SYSTEM_PROMPT_MCQ = """You are an expert math tutor. Solve the multiple-choice math problem accurately.

Select the single best answer from the provided choices.

Reasoning process:
1. Understand the given information and what is being asked.
2. Choose the relevant formula, theorem, or strategy.
3. Solve carefully using exact arithmetic when possible.
4. Compare the result against every answer choice.
5. Choose the equivalent or closest valid option, rounding only when required.

Choice-label rules:
- Choices are labeled A, B, C, etc.
- Output exactly one option label.

Final output rule:
Output ONLY the selected option label inside \\boxed{}.
Do not include explanation, calculations, punctuation, or extra text.
"""

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {str(opt).strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question

def messages_for_item(item, include_answer: bool = False):
    """
    Build valid Qwen chat messages.

    Qwen chat template requires a list like:
    [
      {"role": "system", "content": "..."},
      {"role": "user", "content": "..."},
      {"role": "assistant", "content": "..."}   # optional during SFT
    ]
    """
    question = str(item.get("question", ""))

    options = item.get("options", None)
    if options:
        options = [str(x) for x in options]
    else:
        options = None

    system, user = build_prompt(question, options)

    messages = [
        {"role": "system", "content": str(system)},
        {"role": "user", "content": str(user)},
    ]

    if include_answer:
        answer_text = answer_to_text(item.get("answer", ""))
        messages.append({
            "role": "assistant",
            "content": f"\\boxed{{{answer_text}}}",
        })

    return messages

# ── Model & training config ────────────────────────────────────────────────
MODEL_ID      = 'Qwen/Qwen3-4B-Thinking-2507'
MAX_SEQ_LEN   = 2048
SFT_EPOCHS    = 3
SFT_LR        = 2e-4
GRPO_EPOCHS   = 1
GRPO_LR       = 5e-6

# ── Inference config (A100-optimized) ─────────────────────────────────────
INFER_BATCH_SIZE  = 16
MAX_NEW_TOKENS    = 1024
TEMPERATURE       = 0.6
TOP_P             = 0.95

torch.backends.cuda.matmul.allow_tf32 = True
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


torch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


## Cell 5 — Load dataset

In [ ]:
def normalize_item(item: dict) -> dict:
    """Make every row safe for task-specific Dataset.from_list."""
    item = dict(item)
    item["id"] = int(item.get("id", -1))
    item["question"] = str(item.get("question", ""))

    opts = item.get("options", None)
    item["options"] = [str(x) for x in opts] if opts else []

    raw_answer = item.get("answer", [])
    if isinstance(raw_answer, list):
        answer_list = [str(x) for x in raw_answer]
    elif raw_answer is None:
        answer_list = []
    else:
        answer_list = [str(raw_answer)]

    item["answer_list"] = answer_list
    item["answer_text"] = ", ".join(answer_list)
    item["is_mcq"] = bool(item["options"])
    return item

def load_jsonl_with_preview(path: str) -> List[dict]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Could not find {path}. Run the upload cell and make sure public.jsonl is uploaded."
        )

    rows = []
    with open(path, encoding="utf-8") as f:
        for line in tqdm(f, desc="Loading JSONL"):
            if line.strip():
                rows.append(normalize_item(json.loads(line)))

    n_mcq  = sum(d["is_mcq"] for d in rows)
    n_free = len(rows) - n_mcq
    print(f"Loaded {len(rows)} questions  ({n_mcq} MCQ, {n_free} free-form)")

    mcq_sample  = next((d for d in rows if d["is_mcq"]), None)
    free_sample = next((d for d in rows if not d["is_mcq"]), None)

    if mcq_sample:
        print("\n── MCQ sample ──")
        print(json.dumps({k: mcq_sample[k] for k in ["id", "question", "options", "answer_text"]}, indent=2, ensure_ascii=False))
    if free_sample:
        print("\n── Free-form sample ──")
        print(json.dumps({k: free_sample[k] for k in ["id", "question", "answer_list", "answer_text"]}, indent=2, ensure_ascii=False))

    return rows

data = load_jsonl_with_preview(DATA_PATH)

is_mcq = lambda item: bool(item.get("is_mcq", item.get("options")))
mcq_items  = [x for x in tqdm(data, desc="Separating MCQ") if is_mcq(x)]
free_items = [x for x in tqdm(data, desc="Separating free-form") if not is_mcq(x)]

print(f"\nPrepared {len(mcq_items)} MCQ rows and {len(free_items)} free-form rows.")


Loading JSONL: 0it [00:00, ?it/s]

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "id": 1,
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer_text": "F"
}

── Free-form sample ──
{
  "id": 0,
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer_list": [
    "325*(1+325)"
  ],
  "answer_text": "325*(1+325)"
}


Separating MCQ:   0%|          | 0/1126 [00:00<?, ?it/s]

Separating free-form:   0%|          | 0/1126 [00:00<?, ?it/s]


Prepared 375 MCQ rows and 751 free-form rows.


## Cell 6 — Model loading helpers

In [ ]:
def quant_config():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

def lora_cfg():
    return LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        bias='none', task_type='CAUSAL_LM',
        # Include MLP projections for better reasoning fine-tuning
        target_modules=['q_proj','k_proj','v_proj','o_proj',
                        'gate_proj','up_proj','down_proj'],
    )

def load_tokenizer(src: str):
    tok = AutoTokenizer.from_pretrained(src, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok

def load_qlora_model(model_id: str, adapter_path: Optional[str] = None,
                     padding_side: str = 'right'):
    """Load base model with 4-bit quant + LoRA. Use device_map={'':0}
    (not 'auto') to avoid the set_submodule dispatch bug on older transformers."""
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=quant_config(),
        torch_dtype=torch.bfloat16,
        device_map={'': 0},          # explicit GPU 0 — avoids set_submodule error
        trust_remote_code=True,
        attn_implementation='sdpa',  # scaled-dot-product attention — fast on A100
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)
    if adapter_path and os.path.exists(adapter_path):
        print(f'Loading adapter from {adapter_path}')
        model = PeftModel.from_pretrained(model, adapter_path, is_trainable=True)
    else:
        model = get_peft_model(model, lora_cfg())
    model.print_trainable_parameters()
    return model

def load_inference_model(base_model: str, adapter_dir: Optional[str] = None):
    """Load model for inference (eval mode, padding_side=left for batching)."""
    tok = AutoTokenizer.from_pretrained(
        adapter_dir if adapter_dir else base_model,
        trust_remote_code=True,
    )
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = 'left'  # required for correct batch decoding
    model = AutoModelForCausalLM.from_pretrained(
        base_model,
        quantization_config=quant_config(),
        torch_dtype=torch.bfloat16,
        device_map={'': 0},
        trust_remote_code=True,
        attn_implementation='sdpa',
    )
    if adapter_dir and os.path.exists(adapter_dir):
        print(f'Applying adapter: {adapter_dir}')
        model = PeftModel.from_pretrained(model, adapter_dir)
    else:
        print('No adapter — running base model.')
    model.eval()
    return tok, model

print('Model helpers ready.')

Model helpers ready.


## Cell 7 — SFT training (task-specific adapters)
Trains separate adapters for MCQ and free-form. Skip if already trained.

In [ ]:
import random
from tqdm.auto import tqdm
from datasets import Dataset

def answer_to_text(answer):
    """
    Convert answers only when creating training text.

    MCQ:
      "F" -> "F"

    Free-form:
      ["5/8"] -> "5/8"
      ["143.224229233795", "2.32624773420025"]
        -> "143.224229233795, 2.32624773420025"
    """
    if isinstance(answer, list):
        return ", ".join(str(x) for x in answer)
    return str(answer)


def build_sft_dataset(items: List[dict], tokenizer, max_seq_len: int, task: str):
    """
    Build a Hugging Face Dataset only after tokenization.

    Important:
    Do NOT call Dataset.from_list() on raw JSON rows, because raw rows have:
      - MCQ answer as str
      - free-form answer as list
    """
    examples = []

    for item in tqdm(items, desc=f"Tokenizing SFT {task} examples"):
        full_text = tokenizer.apply_chat_template(
            messages_for_item(item, include_answer=True),
            tokenize=False,
            add_generation_prompt=False,
        )

        prompt_text = tokenizer.apply_chat_template(
            messages_for_item(item, include_answer=False),
            tokenize=False,
            add_generation_prompt=True,
        )

        full = tokenizer(
            full_text,
            truncation=True,
            max_length=max_seq_len,
            padding=False,
        )

        prompt = tokenizer(
            prompt_text,
            truncation=True,
            max_length=max_seq_len,
            padding=False,
        )

        labels = full["input_ids"].copy()
        boundary = min(len(prompt["input_ids"]), len(labels))
        labels[:boundary] = [-100] * boundary

        full["labels"] = labels
        examples.append(full)

    return Dataset.from_list(examples)


def train_sft_task(task_items: List[dict], task: str, tokenizer, output_dir: str):
    print(f"\n===== SFT: {task} ({len(task_items)} rows) =====")

    if not task_items:
        print(f"No {task} rows; skipping SFT.")
        return None

    # Split raw rows with Python lists, not Hugging Face Dataset.
    # This preserves the natural difference in raw data format.
    task_items = list(task_items)
    rng = random.Random(42)
    rng.shuffle(task_items)

    n_eval = max(1, int(len(task_items) * 0.2))
    eval_items = task_items[:n_eval]
    train_items = task_items[n_eval:]

    print(f"Train rows: {len(train_items)} | Eval rows: {len(eval_items)}")

    train_ds = build_sft_dataset(train_items, tokenizer, MAX_SEQ_LEN, task)
    eval_ds = build_sft_dataset(eval_items, tokenizer, MAX_SEQ_LEN, task)
    print(f"Train tokens: {len(train_ds)} | Eval tokens: {len(eval_ds)}")

    out = os.path.join(output_dir, f"{task}_sft_adapter")

    model = load_qlora_model(MODEL_ID)

    collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
        label_pad_token_id=-100,
    )

    targs = TrainingArguments(
        output_dir=out,
        num_train_epochs=SFT_EPOCHS,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=8,
        learning_rate=SFT_LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        max_grad_norm=0.3,
        bf16=True,
        gradient_checkpointing=True,
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        report_to="none",
        remove_unused_columns=False,
    )

    trainer = Trainer(
        model=model,
        args=targs,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=collator,
        processing_class=tokenizer,
    )

    trainer.train()
    trainer.save_model(out)
    tokenizer.save_pretrained(out)

    print(f"Saved {task} SFT adapter -> {out}")

    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()

    return out

# ── Run SFT ───────────────────────────────────────────────────────────────
tokenizer = load_tokenizer(MODEL_ID)
tokenizer.padding_side = 'right'  # right-pad during training

DO_SFT = True   # Set False to skip if adapters already exist
if DO_SFT:
    train_sft_task(free_items, 'free', tokenizer, OUTPUT_DIR)
    train_sft_task(mcq_items,  'mcq',  tokenizer, OUTPUT_DIR)
else:
    print('Skipping SFT.')


===== SFT: free (751 rows) =====
Train rows: 601 | Eval rows: 150


Tokenizing SFT free examples:   0%|          | 0/601 [00:00<?, ?it/s]

Tokenizing SFT free examples:   0%|          | 0/150 [00:00<?, ?it/s]

Train tokens: 601 | Eval tokens: 150


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


Epoch,Training Loss,Validation Loss
1,2.264622,0.207664
2,0.193656,0.204625


Epoch,Training Loss,Validation Loss
1,2.264622,0.207664
2,0.193656,0.204625
3,0.150075,0.203969


Saved mcq SFT adapter -> /content/cse151b_outputs/Qwen/qwen3-task-adapters/mcq_sft_adapter


## Cell 8 — GRPO training (optional, run after SFT)

In [ ]:
_BOX_RE = re.compile(r'\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}', re.DOTALL)

def extract_boxed(text: str) -> str:
    m = _BOX_RE.findall(text or '')
    return m[-1].strip() if m else ''

_BOX_RE = re.compile(r"\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}", re.DOTALL)

def extract_boxed(text: str) -> str:
    matches = _BOX_RE.findall(text or "")
    return matches[-1].strip() if matches else ""


def completion_to_text(c) -> str:
    """
    TRL can pass completions as:
      - plain strings
      - list[dict] chat messages
      - other objects
    Convert safely to text.
    """
    if isinstance(c, str):
        return c

    if isinstance(c, list):
        if len(c) == 0:
            return ""
        last = c[-1]
        if isinstance(last, dict):
            return str(last.get("content", ""))
        return str(last)

    return str(c)


def make_rewards(task: str):
    judger = Judger(strict_extract=False) if Judger else None

    def format_reward(completions, **kwargs):
        results = []

        for c in completions:
            text = completion_to_text(c)
            boxes = _BOX_RE.findall(text)
            one_box = len(boxes) == 1
            has_answer = bool(extract_boxed(text))
            results.append(0.2 if one_box and has_answer else 0.0)

        return results

    def correctness_reward(completions, answer_text=None, answer_list=None, **kwargs):
        results = []

        if answer_text is None:
            answer_text = [""] * len(completions)
        elif not isinstance(answer_text, list):
            answer_text = [answer_text] * len(completions)

        if answer_list is None:
            answer_list = [[x] for x in answer_text]
        elif not isinstance(answer_list, list):
            answer_list = [[answer_list] for _ in completions]

        for c, gold_text, gold_list in zip(completions, answer_text, answer_list):
            text = completion_to_text(c)
            pred = extract_boxed(text)

            if task == "mcq":
                results.append(
                    1.0 if pred.strip().upper() == str(gold_text).strip().upper() else 0.0
                )
            else:
                golds = gold_list if isinstance(gold_list, list) else [gold_text]
                golds = [str(x) for x in golds]

                if judger is None:
                    ok = pred.strip() in [g.strip() for g in golds]
                else:
                    try:
                        ok = judger.auto_judge(
                            pred=text,
                            gold=golds,
                            options=[[]] * len(golds),
                        )
                    except TypeError:
                        # Some Judger versions use positional args.
                        try:
                            ok = judger.auto_judge(text, golds, [[]] * len(golds))
                        except Exception:
                            ok = False
                    except Exception:
                        ok = False

                results.append(1.0 if ok else 0.0)

        return results

    return [correctness_reward, format_reward]

def train_grpo_task(task_items: List[dict], task: str,
                    tokenizer, output_dir: str, sft_start: bool = True):
    if GRPOTrainer is None:
        raise RuntimeError("pip install -U trl first")

    if not task_items:
        print(f"No {task} rows; skipping GRPO.")
        return None

    sft_dir = os.path.join(output_dir, f"{task}_sft_adapter")
    grpo_dir = os.path.join(output_dir, f"{task}_grpo_adapter")
    adapter = sft_dir if sft_start and os.path.exists(sft_dir) else None

    print(f"\n===== GRPO: {task} -> {grpo_dir} =====")
    print(f"Starting from adapter: {adapter}")

    model = load_qlora_model(MODEL_ID, adapter_path=adapter)

    # Split raw rows using Python lists, not Dataset.from_list().
    # This preserves the natural raw data format:
    #   MCQ answer: str
    #   free answer: list
    task_items = list(task_items)
    rng = random.Random(42)
    rng.shuffle(task_items)

    n_eval = max(1, int(len(task_items) * 0.2))
    eval_items = task_items[:n_eval]
    train_items = task_items[n_eval:]

    print(f"Train rows: {len(train_items)} | Eval rows: {len(eval_items)}")

    def to_grpo_rows(items, desc):
        rows = []

        for item in tqdm(items, desc=desc):
            raw_answer = item.get("answer", "")

            if isinstance(raw_answer, list):
                answer_list = [str(x) for x in raw_answer]
                answer_text = ", ".join(answer_list)
            else:
                answer_text = str(raw_answer)
                answer_list = [answer_text]

            rows.append({
                "prompt": messages_for_item(item, include_answer=False),
                "answer_text": answer_text,
                "answer_list": answer_list,
                "options": [str(x) for x in item.get("options", [])],
                "id": int(item.get("id", -1)),
            })

        return rows

    train_rows = to_grpo_rows(train_items, f"Preparing {task} GRPO train")
    eval_rows = to_grpo_rows(eval_items, f"Preparing {task} GRPO eval")

    # Now Dataset.from_list is safe because every row has the same schema.
    train_ds = Dataset.from_list(train_rows)
    eval_ds = Dataset.from_list(eval_rows)

    gargs = GRPOConfig(
        output_dir=grpo_dir,
        learning_rate=GRPO_LR,
        num_train_epochs=GRPO_EPOCHS,

        # Keep num_generations divisible by effective train batch size.
        # With batch_size=1 and grad_accum=4, num_generations=4 is okay.
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        num_generations=2,


        temperature=0.7,
        top_p=0.95,

        beta=0.02,
        loss_type="bnpo",
        use_vllm=False,

        bf16=True,
        gradient_checkpointing=True,
        logging_steps=1,

        save_steps=50,
        save_total_limit=2,
        report_to="none",
        remove_unused_columns=False,
    )

    trainer = GRPOTrainer(
        model=model,
        reward_funcs=make_rewards(task),
        args=gargs,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        processing_class=tokenizer,
        peft_config=None,
    )

    trainer.train()
    trainer.save_model(grpo_dir)
    tokenizer.save_pretrained(grpo_dir)

    print(f"Saved {task} GRPO adapter -> {grpo_dir}")

    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()

    return grpo_dir

DO_GRPO = True   # Change to True only after SFT adapters are trained.
if DO_GRPO:
    train_grpo_task(free_items, 'free', tokenizer, OUTPUT_DIR, sft_start=True)
    train_grpo_task(mcq_items,  'mcq',  tokenizer, OUTPUT_DIR, sft_start=True)
else:
    print('Skipping GRPO.')



===== GRPO: free -> /content/cse151b_outputs/Qwen/qwen3-task-adapters/free_grpo_adapter =====
Starting from adapter: /content/cse151b_outputs/Qwen/qwen3-task-adapters/free_sft_adapter


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Loading adapter from /content/cse151b_outputs/Qwen/qwen3-task-adapters/free_sft_adapter
trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145
Train rows: 601 | Eval rows: 150


Preparing free GRPO train:   0%|          | 0/601 [00:00<?, ?it/s]

...

Preparing mcq GRPO eval:   0%|          | 0/75 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
1,0.000001
2,0.000006
3,0.000012
4,0.000004
5,0.000002
6,0.000003
7,0.000016
8,0.000008
9,0.000005
10,0.000001


Step,Training Loss
1,0.000001
2,0.000006
3,0.000012
4,0.000004
5,0.000002
6,0.000003
7,0.000016
8,0.000008
9,0.000005
10,0.000001


Saved mcq GRPO adapter -> /content/cse151b_outputs/Qwen/qwen3-task-adapters/mcq_grpo_adapter


## Cell 9 — Batched inference
Runs MCQ and free-form subsets with their respective adapters.

In [ ]:
@torch.inference_mode()
def generate_batch(tokenizer, model, prompts: List[str],
                   max_new_tokens: int = MAX_NEW_TOKENS,
                   temperature: float = TEMPERATURE,
                   top_p: float = TOP_P) -> List[str]:
    """Batch generation. tokenizer.padding_side must be 'left' before calling."""
    inputs = tokenizer(
        prompts, return_tensors='pt', padding=True,
        truncation=True, max_length=4096,
    ).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=temperature if temperature > 0 else None,
        top_p=top_p,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )
    # Decode only newly generated tokens, not the prompt
    gen_tokens = out[:, inputs['input_ids'].shape[1]:]
    return tokenizer.batch_decode(gen_tokens, skip_special_tokens=True)

def run_inference_subset(data: List[dict], indices: List[int],
                         task: str, adapter_dir: Optional[str],
                         responses: List[str]):
    if not indices:
        return
    print(f'\n── Inference: {task} ({len(indices)} rows, adapter={adapter_dir}) ──')
    tok, model = load_inference_model(MODEL_ID, adapter_dir)
    prompts = [
        tok.apply_chat_template(
            messages_for_item(data[i], include_answer=False),
            tokenize=False, add_generation_prompt=True,
        )
        for i in indices
    ]
    generated = []
    for j in tqdm(range(0, len(prompts), INFER_BATCH_SIZE), desc=f'{task} batches'):
        batch = prompts[j:j+INFER_BATCH_SIZE]
        generated.extend(generate_batch(tok, model, batch))
    for idx, resp in zip(indices, generated):
        responses[idx] = resp.strip()
    del model, tok; gc.collect(); torch.cuda.empty_cache()

# ── Resolve adapters (prefer grpo, fall back to sft) ──────────────────────
STAGE = 'grpo'   # change to 'sft' to use SFT adapters
fallback = 'sft' if STAGE == 'grpo' else 'grpo'

def resolve_adapter(task):
    for stage in [STAGE, fallback]:
        p = os.path.join(OUTPUT_DIR, f'{task}_{stage}_adapter')
        if os.path.exists(p):
            return p
    return None   # base model only

free_adapter = resolve_adapter('free')
mcq_adapter  = resolve_adapter('mcq')
print(f'Free adapter: {free_adapter}')
print(f'MCQ  adapter: {mcq_adapter}')

free_indices = [i for i, x in enumerate(data) if not is_mcq(x)]
mcq_indices  = [i for i, x in enumerate(data) if is_mcq(x)]
responses = [''] * len(data)

run_inference_subset(data, free_indices, 'free-form', free_adapter, responses)
run_inference_subset(data, mcq_indices,  'MCQ',       mcq_adapter,  responses)
print(f'\nGenerated {sum(bool(r) for r in responses)}/{len(responses)} responses.')

Free adapter: /content/cse151b_outputs/Qwen/qwen3-task-adapters/free_grpo_adapter
MCQ  adapter: /content/cse151b_outputs/Qwen/qwen3-task-adapters/mcq_grpo_adapter

── Inference: free-form (751 rows, adapter=/content/cse151b_outputs/Qwen/qwen3-task-adapters/free_grpo_adapter) ──


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Applying adapter: /content/cse151b_outputs/Qwen/qwen3-task-adapters/free_grpo_adapter


free-form batches:   0%|          | 0/47 [00:00<?, ?it/s]


── Inference: MCQ (375 rows, adapter=/content/cse151b_outputs/Qwen/qwen3-task-adapters/mcq_grpo_adapter) ──


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Applying adapter: /content/cse151b_outputs/Qwen/qwen3-task-adapters/mcq_grpo_adapter


MCQ batches:   0%|          | 0/24 [00:00<?, ?it/s]


Generated 1126/1126 responses.


## Cell 10 — Score & save results

In [36]:
BOX_RE = re.compile(r'\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}', re.DOTALL)

def extract_boxed(text: str) -> str:
    m = BOX_RE.findall(text or '')
    return m[-1].strip() if m else ''

def score_all(data, responses):
    if Judger is None:
        print('Judger unavailable — skipping free-form scoring.')
    judger = Judger(strict_extract=False) if Judger else None
    results = []
    for item, response in tqdm(zip(data, responses), total=len(data), desc='Scoring'):
        mcq  = is_mcq(item)
        gold = item.get('answer_text')
        if gold is None:
            correct = None
        elif mcq:
            correct = extract_boxed(response).upper() == str(gold).strip().upper()
        else:
            gold_list = item.get('answer_list') or ([gold] if gold else [])
            if judger is None:
                correct = None
            else:
                try:   correct = judger.auto_judge(response, [str(x) for x in gold_list], [[]]*len(gold_list))
                except: correct = False
        results.append({'id': item.get('id'), 'is_mcq': mcq, 'gold': item.get('answer_list') if not mcq else gold,
                        'response': response, 'correct': correct})
    return results

results = score_all(data, responses)

# ── Print summary ──────────────────────────────────────────────────────────
valid    = [r for r in results if r['correct'] is not None]
mcq_res  = [r for r in valid if r['is_mcq']]
free_res = [r for r in valid if not r['is_mcq']]
acc = lambda xs: 100*sum(bool(r['correct']) for r in xs)/len(xs) if xs else 0.0

print('='*55)
print('EVALUATION RESULTS')
print(f"  MCQ       : {sum(r['correct'] for r in mcq_res):4d}/{len(mcq_res):4d} ({acc(mcq_res):.2f}%)")
print(f"  Free-form : {sum(r['correct'] for r in free_res):4d}/{len(free_res):4d} ({acc(free_res):.2f}%)")
print(f"  Overall   : {sum(r['correct'] for r in valid):4d}/{len(valid):4d} ({acc(valid):.2f}%)")
print('='*55)

# ── Save JSONL ────────────────────────────────────────────────────────────
out_path = os.path.join(RESULTS_DIR, 'public_eval_results.jsonl')
with open(out_path, 'w', encoding='utf-8') as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')
print(f'Saved {len(results)} records -> {out_path}')

# ── Save CSV for submission ────────────────────────────────────────────────
import csv
csv_path = os.path.join(RESULTS_DIR, 'submission.csv')
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['id','response'])
    w.writeheader()
    for r in results:
        w.writerow({'id': r['id'], 'response': r['response']})
print(f'Saved submission CSV -> {csv_path}')

Scoring:   0%|          | 0/1126 [00:00<?, ?it/s]

EVALUATION RESULTS
  MCQ       :  183/ 375 (48.80%)
  Free-form :  298/ 751 (39.68%)
  Overall   :  481/1126 (42.72%)
Saved 1126 records -> /content/cse151b_outputs/results/public_eval_results.jsonl
Saved submission CSV -> /content/cse151b_outputs/results/submission.csv
